In [1]:
!pip install optuna mlflow dagshub seaborn

In [2]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import dagshub
import mlflow
import mlflow.sklearn

In [3]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [4]:
df = pd.read_csv("twitter_cleaned.csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [5]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

# Convert to numpy (prevents indexing errors in CV)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [6]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (41292, 8000)
Test shape: (10323, 8000)


In [7]:
def build_logreg(trial):

    solver = trial.suggest_categorical("solver", ["lbfgs", "saga"])
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)

    return LogisticRegression(
        C=C,
        solver=solver,
        penalty="l2",
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

In [8]:
def objective(trial):

    model = build_logreg(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(trial.params)
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [9]:
study = optuna.create_study(
    direction="maximize",
    study_name="logreg_study",
    storage="sqlite:///logreg_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 08:40:17,242] A new study created in RDB with name: logreg_study


In [10]:
with mlflow.start_run(run_name="Logistic_Regression"):

    mlflow.log_param("model_name", "Logistic_Regression")
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 8000)
    mlflow.log_param("min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = LogisticRegression(
        **best_params,
        penalty="l2",
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X_train, y_train)

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = LogisticRegression(
            **best_params,
            penalty="l2",
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_logreg.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_logreg.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - Logistic Regression")
    plt.savefig("confusion_matrix_logreg.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_logreg.png")

    study.trials_dataframe().to_csv("optuna_trials_logreg.csv", index=False)
    mlflow.log_artifact("optuna_trials_logreg.csv")

    mlflow.sklearn.log_model(final_model, artifact_path="model")

print("Logistic Regression experiment completed successfully.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[I 2026-02-23 08:41:00,655] Trial 0 finished with value: 0.7599185813232952 and parameters: {'solver': 'saga', 'C': 9.886381788944982}. Best is trial 0 with value: 0.7599185813232952.


🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/b44c51f3b7f3435db7edad10380eb52a
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 08:41:04,953] Trial 1 finished with value: 0.7975498684957317 and parameters: {'solver': 'lbfgs', 'C': 0.7987842912493428}. Best is trial 1 with value: 0.7975498684957317.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/0ce0a68e322e4ef6ad33a0113b062f1e
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 08:41:35,121] Trial 2 finished with value: 0.7264310322226354 and parameters: {'solver': 'saga', 'C': 0.07024424578581055}. Best is trial 1 with value: 0.7975498684957317.


...

🏃 View run trial_48 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/cb3e978965824e09b80efd006e3aa694
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
🏃 View run trial_49 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/fbdc3207289544a0b263699bcf3f04d7
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 08:51:12,914] Trial 49 finished with value: 0.6383971014434661 and parameters: {'solver': 'lbfgs', 'C': 0.002333923948505282}. Best is trial 41 with value: 0.8084710345731646.


Best Macro F1: 0.8084710345731646
Best Params: {'solver': 'lbfgs', 'C': 3.288839315142639}
Fold 1 - Macro F1: 0.7982
Fold 2 - Macro F1: 0.7968
Fold 3 - Macro F1: 0.7929
Fold 4 - Macro F1: 0.8008
Fold 5 - Macro F1: 0.7947
Final Test Macro F1: 0.8084710345731646


2026/02/23 08:51:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 08:51:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic_Regression at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/c29d96f80b884def92c60ffe742ee35e
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
Logistic Regression experiment completed successfully.
